## 1. Project Initialization

### Dependencies

Before we begin, we'll install the required libraries for this RAG pipeline:

- **langchain**: Core LangChain framework (v0.2.17)
- **langchain-community**: Community integrations including FAISS
- **google-generativeai**: Google Gemini LLM integration
- **sentence-transformers**: HuggingFace embeddings models
- **faiss-cpu**: Vector database for similarity search
- **pypdf**: PDF document loader
- **python-dotenv**: Environment variable management
- **ipywidgets**: Interactive Jupyter widgets for UI

Let's install these dependencies:

In [1]:
import subprocess
import sys

# First, upgrade pip to avoid version conflicts
print("Upgrading pip...")
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], capture_output=True)

# Install compatible versions to avoid import errors
print("\nInstalling compatible LangChain versions...")
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "langchain==0.2.17", "langchain-core==0.2.43", "langchain-community==0.2.19"], capture_output=True)

# Force uninstall and reinstall latest google-generativeai
print("Uninstalling old google-generativeai (if any)...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "google-generativeai"], capture_output=True)
print("Installing latest google-generativeai...")
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "google-generativeai"], capture_output=True)

# Install required packages
packages = [
    "faiss-cpu",
    "pypdf",
    "python-dotenv",
    "sentence-transformers",
    "ipywidgets",
    "typing-extensions"
 ]

print("\nInstalling required packages...")
print(f"Python executable: {sys.executable}\n")

failed_packages = []

for package in packages:
    print(f"Installing {package}...", end=" ")
    try:
        result = subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", package], capture_output=True, text=True, timeout=120)
        if result.returncode == 0:
            print("✓ Success")
        else:
            print("✗ Failed")
            print(f"  Error: {result.stderr[:200]}")
            failed_packages.append((package, result.stderr))
    except subprocess.TimeoutExpired:
        print("✗ Timeout")
        failed_packages.append((package, "Installation timed out after 120 seconds"))
    except Exception as e:
        print(f"✗ Exception: {str(e)}")
        failed_packages.append((package, str(e)))

print("\n" + "="*70)
if failed_packages:
    print(f"⚠️  {len(failed_packages)} package(s) failed to install:")
    for pkg, error in failed_packages:
        print(f"\n  ❌ {pkg}")
        print(f"     Error: {error[:200]}")
else:
    print("✓ All packages installed successfully!")
print("="*70)

Upgrading pip...

Installing compatible LangChain versions...
Uninstalling old google-generativeai (if any)...
Installing latest google-generativeai...

Installing required packages...
Python executable: c:\Users\Yash Sinha\Desktop\Projects\Projects\python project\.venv\Scripts\python.exe

Installing faiss-cpu... ✓ Success
Installing pypdf... ✓ Success
Installing python-dotenv... ✓ Success
Installing sentence-transformers... ✓ Success
Installing ipywidgets... ✓ Success
Installing typing-extensions... ✓ Success

✓ All packages installed successfully!


## 2. Environment & Configuration

### Secure API Key Management

We use `python-dotenv` to load environment variables from a `.env` file. This approach:

- Keeps sensitive credentials out of version control
- Allows different configurations per environment
- Prevents accidental API key exposure

Before running this notebook, ensure your `.env` file contains a valid `GOOGLE_API_KEY` for Google Gemini and optionally configure HuggingFace embeddings.

Let's load and validate the configuration:

In [2]:
import os
try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv"])
    from dotenv import load_dotenv
from pathlib import Path

# Load environment variables from .env file
env_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path)

# Retrieve configuration
GOOGLE_API_KEY: str = os.getenv("GOOGLE_API_KEY", "")
PDF_PATH: str = os.getenv("PDF_DATA_PATH", "./data/sample.pdf")
FAISS_INDEX_PATH: str = os.getenv("FAISS_INDEX_PATH", "./faiss_index")
CHUNK_SIZE: int = int(os.getenv("CHUNK_SIZE", "1000"))
CHUNK_OVERLAP: int = int(os.getenv("CHUNK_OVERLAP", "100"))
TOP_K_RESULTS: int = int(os.getenv("TOP_K_RESULTS", "3"))
EMBEDDING_MODEL: str = os.getenv("EMBEDDING_MODEL", "all-MiniLM-L6-v2")

# Validate API key
if not GOOGLE_API_KEY:
    raise ValueError(
        "❌ GOOGLE_API_KEY not found in .env file. "
        "Please set it before proceeding."
    )

print("✓ Configuration loaded successfully!")
print(f"  - PDF Path: {PDF_PATH}")
print(f"  - FAISS Index Path: {FAISS_INDEX_PATH}")
print(f"  - Chunk Size: {CHUNK_SIZE}")
print(f"  - Chunk Overlap: {CHUNK_OVERLAP}")
print(f"  - Top K Results: {TOP_K_RESULTS}")
print(f"  - Embedding Model: {EMBEDDING_MODEL}")

✓ Configuration loaded successfully!
  - PDF Path: C:\Users\Yash Sinha\Desktop\Projects\Projects\python project\NIPS-2017-attention-is-all-you-need-Paper.pdf
  - FAISS Index Path: ./faiss_index
  - Chunk Size: 1000
  - Chunk Overlap: 100
  - Top K Results: 3
  - Embedding Model: all-MiniLM-L6-v2


## 3. Document Ingestion

### Loading PDF Documents

We use `PyPDFLoader` from LangChain to extract text content from PDF files. This loader:

- Reads PDF files from a specified path
- Extracts text from each page
- Preserves metadata (source, page number)
- Returns a list of `Document` objects

**Note:** Ensure your PDF file exists at the path specified in the `.env` file.

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from typing import List

def load_pdf_documents(pdf_path: str) -> List[Document]:
    """
    Load documents from a PDF file.
    
    Args:
        pdf_path: Path to the PDF file
        
    Returns:
        List of Document objects with text and metadata
        
    Raises:
        FileNotFoundError: If PDF file doesn't exist
        Exception: If PDF loading fails
    """
    try:
        if not Path(pdf_path).exists():
            raise FileNotFoundError(f"PDF file not found at: {pdf_path}")
        
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        
        print(f"✓ Loaded {len(documents)} pages from PDF")
        return documents
    
    except FileNotFoundError as e:
        print(f"❌ {e}")
        print(f"   Please ensure the PDF exists at: {pdf_path}")
        raise
    except Exception as e:
        print(f"❌ Error loading PDF: {str(e)}")
        raise

# Load documents (comment out if PDF doesn't exist yet)
try:
    raw_documents = load_pdf_documents(PDF_PATH)
    print(f"\n📄 First page preview (first 250 chars):")
    print(raw_documents[0].page_content[:250] if raw_documents else "No content")
except (FileNotFoundError, NameError) as e:
    print(f"⚠️  PDF not available: {str(e)}")
    print("   Please check PDF_PATH and ensure the file exists.")

✓ Loaded 11 pages from PDF

📄 First page preview (first 250 chars):
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@goo


## 4. Optimal Text Splitting
We use `RecursiveCharacterTextSplitter` which:
- Splits on natural boundaries (paragraphs, sentences, words)
- Maintains semantic coherence better than naive splitting
- Supports chunk size and overlap parameters

In [4]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_documents(
    documents: List[Document],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP
) -> List[Document]:
    """
    Split documents into chunks with overlap.
    
    Args:
        documents: List of Document objects to split
        chunk_size: Maximum size of each chunk
        chunk_overlap: Number of characters to overlap between chunks
        
    Returns:
        List of split Document objects with metadata
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]  # Progressive splitting strategy
    )
    
    split_docs = splitter.split_documents(documents)
    return split_docs

# Split documents
try:
    split_documents_list = split_documents(raw_documents)
    print(f"✓ Split into {len(split_documents_list)} chunks")
    print(f"\n  📊 Chunk Statistics:")
    print(f"     - Total chunks: {len(split_documents_list)}")
    print(f"     - Chunk size range: {min(len(d.page_content) for d in split_documents_list)}-{max(len(d.page_content) for d in split_documents_list)} chars")
    print(f"\n  Sample chunk (first 200 chars):")
    print(f"     {split_documents_list[0].page_content[:200]}...")
except NameError:
    print("⚠️  raw_documents not available. Ensure PDF was loaded successfully.")

✓ Split into 40 chunks

  📊 Chunk Statistics:
     - Total chunks: 40
     - Chunk size range: 185-997 chars

  Sample chunk (first 200 chars):
     Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz...


## 5. Vectorization & Persistence (Direct Approach)

### Embeddings: Converting Text to Vectors with SentenceTransformer

We use **SentenceTransformer directly** for fine-grained control:

- **Model**: HuggingFace's Sentence Transformers (all-MiniLM-L6-v2 by default)
  - Dimensions: 384
  - Lightweight, runs locally without API calls
  - Excellent semantic similarity performance
  - Privacy-preserving (no data sent externally)

- **Direct Encoding**: Batch encode all documents at once
  - Full numpy array output
  - Progress bar for large datasets
  - Direct control over encoding parameters

### Vector Database: FAISS with Low-Level Operations

FAISS (Facebook AI Similarity Search):
- **IndexFlatL2**: Exact L2 distance computation for highest accuracy
- **Direct Index Operations**: 
  - Direct `.add()` to insert vectors
  - Direct `.search()` for similarity queries
  - Full control over distance metrics

- **Persistent Storage**:
  - Save/load FAISS index directly with `faiss.write_index()` and `faiss.read_index()`
  - Save embeddings as numpy arrays
  - Store documents as JSON for reconstruction

This approach provides maximum control and transparency over the RAG pipeline.

In [5]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

def create_embeddings_and_faiss_index(documents: List[Document]) -> tuple:
    """
    Create embeddings using SentenceTransformer and build FAISS index directly.
    
    Args:
        documents: List of split Document objects
        
    Returns:
        Tuple of (index, embeddings, documents, model)
    """
    try:
        print(f"🔄 Loading SentenceTransformer model ({EMBEDDING_MODEL})...")
        model = SentenceTransformer(EMBEDDING_MODEL)
        
        # Extract text from documents
        doc_texts = [doc.page_content for doc in documents]
        
        print(f"🔄 Generating embeddings for {len(doc_texts)} documents...")
        embeddings = model.encode(doc_texts, show_progress_bar=True, convert_to_numpy=True)
        
        # Create FAISS index
        print(f"🔄 Creating FAISS index...")
        dim = embeddings.shape[1]
        index = faiss.IndexFlatL2(dim)
        index.add(embeddings.astype(np.float32))
        
        print(f"✓ FAISS index created successfully!")
        print(f"   - Embedding dimension: {dim}")
        print(f"   - Number of vectors: {index.ntotal}")
        
        return index, embeddings, documents, model
        
    except Exception as e:
        print(f"❌ Error creating embeddings and FAISS index: {str(e)}")
        raise

def save_fast_index(index: faiss.IndexFlatL2, embeddings: np.ndarray, documents: List[Document], path: str) -> None:
    """
    Save FAISS index and embeddings to disk.
    
    Args:
        index: FAISS index
        embeddings: Numpy array of embeddings
        documents: List of documents
        path: Directory path to save
    """
    try:
        Path(path).mkdir(parents=True, exist_ok=True)
        faiss.write_index(index, str(Path(path) / "faiss_index.bin"))
        np.save(str(Path(path) / "embeddings.npy"), embeddings)
        
        # Save documents as JSON for reference
        import json
        docs_data = [
            {
                "text": doc.page_content,
                "metadata": doc.metadata
            }
            for doc in documents
        ]
        with open(Path(path) / "documents.json", "w") as f:
            json.dump(docs_data, f, indent=2)
        
        print(f"✓ FAISS index and embeddings saved to: {path}")
    except Exception as e:
        print(f"❌ Error saving FAISS index: {str(e)}")
        raise

# Create embeddings and FAISS index
try:
    faiss_index, doc_embeddings, indexed_documents, embedding_model = create_embeddings_and_faiss_index(split_documents_list)
    save_fast_index(faiss_index, doc_embeddings, indexed_documents, FAISS_INDEX_PATH)
except NameError:
    print("⚠️  split_documents_list not available. Ensure previous cells executed successfully.")

🔄 Loading SentenceTransformer model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Generating embeddings for 40 documents...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

🔄 Creating FAISS index...
✓ FAISS index created successfully!
   - Embedding dimension: 384
   - Number of vectors: 40
✓ FAISS index and embeddings saved to: ./faiss_index


## 6. Rehydration & Direct Retrieval

### Loading Persisted FAISS Index

Load the pre-computed embeddings and FAISS index from disk for production use without rebuilding:

- **FAISS Index** (`faiss_index.bin`): Optimized search structure
- **Embeddings** (`embeddings.npy`): Pre-computed document vectors  
- **Documents** (`documents.json`): Original text + metadata

This enables fast retrieval in new sessions.

In [6]:
import json

def load_faiss_and_embeddings(path: str) -> tuple:
    """
    Load FAISS index, embeddings, and documents from disk.
    
    Args:
        path: Directory path where FAISS index is stored
        
    Returns:
        Tuple of (index, embeddings, documents, model)
    """
    try:
        if not Path(path).exists():
            raise FileNotFoundError(f"FAISS index not found at: {path}")
        
        print(f"🔄 Loading FAISS index and embeddings from {path}...")
        
        # Load FAISS index
        index = faiss.read_index(str(Path(path) / "faiss_index.bin"))
        
        # Load embeddings
        embeddings = np.load(str(Path(path) / "embeddings.npy"))
        
        # Load documents
        with open(Path(path) / "documents.json", "r") as f:
            docs_data = json.load(f)
        
        # Reconstruct Document objects using langchain_core.documents
        documents = [
            Document(
                page_content=doc["text"],
                metadata=doc["metadata"]
            )
            for doc in docs_data
        ]
        
        # Load embedding model
        model = SentenceTransformer(EMBEDDING_MODEL)
        
        print(f"✓ FAISS index and embeddings loaded successfully!")
        print(f"   - Number of vectors: {index.ntotal}")
        print(f"   - Embedding dimension: {embeddings.shape[1]}")
        print(f"   - Number of documents: {len(documents)}")
        
        return index, embeddings, documents, model
        
    except FileNotFoundError as e:
        print(f"❌ {e}")
        raise
    except Exception as e:
        print(f"❌ Error loading FAISS index: {str(e)}")
        raise

# Load the persisted index
try:
    faiss_idx, doc_embeds, doc_list, embed_model = load_faiss_and_embeddings(FAISS_INDEX_PATH)
    print("\n✅ Ready for retrieval and Q&A!")
except FileNotFoundError as e:
    print(f"⚠️  Cannot load index yet. Run previous cells to create the index first.")

🔄 Loading FAISS index and embeddings from ./faiss_index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ FAISS index and embeddings loaded successfully!
   - Number of vectors: 40
   - Embedding dimension: 384
   - Number of documents: 40

✅ Ready for retrieval and Q&A!


## 7. Direct Retrieval Functions

### FAISS Vector Search

Retrieve relevant document chunks using L2 distance similarity search:

- **Query Encoding**: Encode user question with SentenceTransformer
- **L2 Distance Search**: Find nearest vectors in FAISS index
- **Top-K Retrieval**: Return most relevant document chunks with metadata

This provides fast, transparent semantic search.

In [7]:
def retrieve_top_k_chunks(query: str, k: int = TOP_K_RESULTS) -> List[Document]:
    """
    Retrieve top-K relevant document chunks using FAISS search.
    
    Args:
        query: User's question
        k: Number of top results
        
    Returns:
        List of top-K Document objects
    """
    try:
        # Encode query
        query_vector = embed_model.encode([query], show_progress_bar=False).astype(np.float32)
        
        # Search FAISS index (L2 distance)
        distances, indices = faiss_idx.search(query_vector, k)
        
        # Retrieve documents
        retrieved_docs = [doc_list[idx] for idx in indices[0] if idx < len(doc_list)]
        
        return retrieved_docs
        
    except Exception as e:
        print(f"❌ Error retrieving chunks: {str(e)}")
        return []

## 8. Google Gemini LLM Integration

### Combining Retrieval with Generative AI

The complete RAG pipeline:

1. **Retrieve**: Get relevant document chunks from FAISS
2. **Augment**: Build context from retrieved documents
3. **Generate**: Use Google Gemini to answer based on context

Google Gemini Features:
- Fast response generation
- Excellent at following instructions
- Good context understanding
- Paper summary and detail analysis capabilities

In [8]:
import google.generativeai as genai

# Configure Google Gemini API
genai.configure(api_key=GOOGLE_API_KEY)

def generate_summary(documents: List[Document]) -> str:
    """
    Generate a single-line summary of the PDF content.
    
    Args:
        documents: List of all documents
        
    Returns:
        One-line summary string
    """
    try:
        model = genai.GenerativeModel(model_name="gemini-2.5-flash-lite")
        
        # Combine all document content and clean it
        full_text = "\n\n".join([doc.page_content for doc in documents]).strip()
        if not full_text:
            return "No content found in the PDF to summarize."
        # Keep a larger slice for better summary quality
        full_text = full_text[:6000]
        
        prompt = (
            "Summarize the following PDF content in 1-2 sentences. "
            "Be specific and avoid generic phrases.\n\n"
            f"{full_text}\n\n"
            "Summary:"
        )
        
        response = model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(
                temperature=0.3,
                max_output_tokens=180
            )
        )
        
        text = getattr(response, "text", "")
        return text.strip() if text else "Unable to generate summary"
        
    except Exception as e:
        print(f"❌ Error generating summary: {str(e)}")
        return "Unable to generate summary"

def answer_question(query: str, context_chunks: List[Document], conversation_history: List = None) -> str:
    """
    Answer a question using retrieved context with Gemini.
    
    Args:
        query: User's question
        context_chunks: Retrieved document chunks
        conversation_history: Previous Q&A for context (optional)
        
    Returns:
        Generated answer string
    """
    try:
        model = genai.GenerativeModel(model_name="gemini-2.5-flash-lite")
        
        # Build context string
        context_text = "\n\n".join([
            f"[{doc.metadata.get('source', 'Unknown')} - Page {doc.metadata.get('page', 'N/A')}]\n{doc.page_content}"
            for doc in context_chunks
        ])
        
        # Build prompt with conversation history
        history_text = ""
        if conversation_history:
            history_text = "Previous questions asked:\n"
            for q, a in conversation_history[-3:]:  # Last 3 Q&A pairs
                history_text += f"Q: {q}\nA: {a[:100]}...\n\n"
        
        prompt = f"""You are a helpful PDF assistant. Answer questions based ONLY on the provided context.

{history_text}

CONTEXT FROM PDF:
{context_text}

QUESTION:
{query}

ANSWER:"""
        
        response = model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(
                temperature=0.7,
                max_output_tokens=500
            )
        )
        
        text = getattr(response, "text", "")
        return text.strip() if text else "Unable to generate answer. Please try again."
        
    except Exception as e:
        print(f"❌ Error generating answer: {str(e)}")
        return "Unable to generate answer. Please try again."

def retrieve_and_answer(query: str, conversation_history: List = None) -> dict:
    """
    Complete RAG pipeline: retrieve documents and generate answer.
    
    Args:
        query: User's question
        conversation_history: Previous Q&A pairs
        
    Returns:
        Dictionary with answer, sources, and metadata
    """
    try:
        # Retrieve relevant chunks
        retrieved_chunks = retrieve_top_k_chunks(query, TOP_K_RESULTS)
        
        if not retrieved_chunks:
            return {
                "answer": "Could not find relevant information in the PDF.",
                "sources": [],
                "success": False
            }
        
        # Generate answer
        answer = answer_question(query, retrieved_chunks, conversation_history)
        
        # Extract sources
        sources = [
            {
                "page": doc.metadata.get("page", "N/A"),
                "source": doc.metadata.get("source", "Unknown")
            }
            for doc in retrieved_chunks
        ]
        
        return {
            "answer": answer,
            "sources": sources,
            "retrieved_count": len(retrieved_chunks),
            "success": True
        }
        
    except Exception as e:
        print(f"❌ Error in RAG pipeline: {str(e)}")
        return {
            "answer": f"Error: {str(e)}",
            "sources": [],
            "success": False
        }

print("✅ Google Gemini LLM integration ready!")

✅ Google Gemini LLM integration ready!


C:\Users\Yash Sinha\AppData\Local\Temp\ipykernel_9104\4075206500.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 9. Interactive PDF Q&A Interface

### PDF Summary + Question Box

Get a quick overview of the PDF and ask questions with follow-up support:

- **Single-Line Summary**: Understand the PDF at a glance
- **Persistent Input Box**: Ask questions and get answers
- **Follow-up Support**: Ask follow-up questions on the same topic
- **Source Citations**: See which pages the answers came from

Run the cell below to start interacting with your PDF!

In [9]:
from IPython.display import display, HTML, Markdown, clear_output
import ipywidgets as widgets

# Initialize conversation history
conversation_history = []

# Validate required functions and data before UI
missing_items = []
for name in ["generate_summary", "retrieve_and_answer", "doc_list"]:
    if name not in globals():
        missing_items.append(name)
if missing_items:
    print("⚠️  Missing required items:", ", ".join(missing_items))
    print("   Please run all previous setup cells (PDF load, embeddings, Gemini integration) first.")
else:
    try:
        # Step 1: Generate Summary
        print("\n" + "="*70)
        print("📄 PDF INTELLIGENT Q&A INTERFACE")
        print("="*70)
        
        print("\n🔄 Generating PDF summary...")
        pdf_summary = generate_summary(doc_list)
        
        print("\n" + "="*70)
        print("📌 PDF SUMMARY")
        print("="*70)
        print(f"\n{pdf_summary}\n")
        
        # Step 2: Create interactive Q&A interface
        print("="*70)
        print("💬 QUESTION & ANSWER")
        print("="*70)
        
        # Create question input box
        question_input = widgets.Text(
            placeholder='Ask a question about the PDF...',
            description='Q:',
            style={'description_width': '30px'},
            layout=widgets.Layout(width='100%', height='50px')
        )
        
        # Create submit button
        submit_button = widgets.Button(
            description='Send',
            button_style='info',
            tooltip='Submit your question',
            icon='send'
        )
        
        # Create output area for answers
        answer_output = widgets.Output(layout=widgets.Layout(
            border='1px solid #ccc',
            padding='10px',
            margin='10px 0',
            min_height='100px'
        ))
        
        # Create new question button (shown after first answer)
        new_question_button = widgets.Button(
            description='Ask Another Question',
            button_style='success',
            tooltip='Ask a follow-up question',
            icon='plus'
        )
        
        # Track question count
        question_count = [0]
        
        def on_submit_clicked(b):
            question_text = question_input.value.strip()
            
            if not question_text:
                with answer_output:
                    print("⚠️  Please enter a question first.")
                return
            
            # Disable input while processing
            submit_button.disabled = True
            question_input.disabled = True
            question_input.value = ""
            
            with answer_output:
                clear_output()
                print(f"🤖 Processing: '{question_text}'...\n")
                
                # Get RAG response
                result = retrieve_and_answer(question_text, conversation_history)
                
                if result["success"]:
                    # Add to conversation history
                    conversation_history.append((question_text, result["answer"]))
                    question_count[0] += 1
                    
                    print("✅ ANSWER:")
                    print("-" * 70)
                    print(result["answer"])
                    print("-" * 70)
                    print(f"\n📚 Sources ({result['retrieved_count']} documents):")
                    for i, source in enumerate(result["sources"], 1):
                        print(f"   [{i}] {source['source']} - Page {source['page']}")
                    
                    # Show follow-up prompt
                    print("\n💡 Ask a follow-up question or a new question above!")
                else:
                    print(f"❌ Error: {result['answer']}")
            
            # Re-enable input
            submit_button.disabled = False
            question_input.disabled = False
            question_input.focus()
        
        def on_new_question_clicked(b):
            # Just focus the input - user can type new question
            question_input.focus()
        
        # Attach event handlers
        submit_button.on_click(on_submit_clicked)
        new_question_button.on_click(on_new_question_clicked)
        
        # Display interface
        print("\n📝 Enter your question below and click 'Send':\n")
        
        # Create HBox for input and button
        input_box = widgets.HBox([question_input, submit_button], 
                                 layout=widgets.Layout(gap='10px'))
        
        display(input_box)
        display(answer_output)
        
        # Display follow-up button info
        print("\n✨ Keep asking follow-up questions to explore more about the PDF!")
        print("   The system will maintain context from previous answers.\n")
        
    except Exception as e:
        print(f"❌ Error initializing Q&A interface: {str(e)}")
        import traceback
        traceback.print_exc()


📄 PDF INTELLIGENT Q&A INTERFACE

🔄 Generating PDF summary...

📌 PDF SUMMARY

The Transformer is a novel network architecture that replaces traditional recurrent and convolutional layers with attention mechanisms, achieving superior quality and faster training times for machine translation tasks. This model demonstrates state-of-the-art performance on English-to-German and English-to-French translation benchmarks, requiring significantly less computational resources.

💬 QUESTION & ANSWER

📝 Enter your question below and click 'Send':



Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…


✨ Keep asking follow-up questions to explore more about the PDF!
   The system will maintain context from previous answers.

